In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *
from biked_commons.conditioning import conditioning

In [ ]:
def sample_continuous(num_samples, split="test", randomize = False):
    emb = conditioning.sample_image_embedding(num_samples, split, randomize)
    rider = conditioning.sample_riders(num_samples, split, randomize)
    use_case = conditioning.sample_use_case(num_samples, split, randomize)
    all = torch.cat((emb, rider, use_case), dim=1)
    return all

def parse_continuous_condition(condition):
    image_embeddings = condition[:, :512]
    use_case_condition = condition[:, -3:]
    rider_condition = condition[:, 512:-3]
    condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
    return condition


In [ ]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)
data_tens = torch.tensor(data.values, dtype=torch.float32)

evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)

ref_point = get_ref_point(evaluator, requirement_names)

def calc_composite_score(data_tens, condition, evaluator = evaluator):):
    # Calculate the composite score for each row in the data tensor
    eval_scores = evaluator(data_tens, condition)
    